# 🎬 Discipline Prime — a real AI film on Kaggle 2×T4

**One 3-minute film, real engines, no text animation.** The factory in this repo now renders
its shots with **LTX-Video 2B text-to-video diffusion — one worker per T4, both GPUs in
parallel** (any shot diffusion can't finish falls back to an AI-still camera move, so the
film always ships), then conforms every shot to the film's clock and finishes with the
repo's own audio stack:

> Groq director (validated shot-prompt polish) → edge-tts narration → adaptive timing to
> exactly 180.0 s → code-composed score + synthesized foley → 3-bus mix at −14 LUFS →
> **LTX-Video shots on both T4s** → conform + dip-to-black chapters + grade → NVENC
> (auto) master → mux → `bin/vh check` gates.

**Requirements (set before running):** Accelerator **GPU T4 x2** · Internet **ON**
(edge-tts, Groq, HF model download ~10 GB, pip).

Runtime estimate: ~60–120 min (model download ~5 min, 36 shots × ~1–3 min across two GPUs,
audio minutes). Outputs land in `/kaggle/working/discipline-films/`.

## 1 · Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG — one cell to rule the whole run
# ══════════════════════════════════════════════════════════════════════════

GROQ_API_KEY = ""            # ← Kaggle Secret GROQ_API_KEY wins; inline is the private fallback

GROQ_TEXT_MODEL = "auto"     # checked against the live /models at boot
TEXT_MODEL_PREFERENCE = [
    "openai/gpt-oss-120b", "llama-3.3-70b-versatile", "openai/gpt-oss-20b",
    "qwen/qwen3-32b", "llama-3.1-8b-instant",
]
VISION_MODEL_PREFERENCE = [
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "meta-llama/llama-4-maverick-17b-128e-instruct",
]

ENGINE = "auto"              # auto | diffusion | kenburns  (auto = diffusion when CUDA is visible)
TARGET_SECONDS = 180.0       # the film blueprint (180.0 s) wins if it disagrees
FINAL_TIER = "final"         # final = 1080p master | beast = 720p
AI_DIRECTOR = True           # Groq shot-prompt polish + metadata (validated, blueprint fallback)
DRAFT_VISION_QA = True       # contact sheet + Groq vision rubric on the finished master

## 2 · Boot — the repo, the GPUs, the diffusion stack

In [ ]:
import os, subprocess

def run(cmd, cwd=None, timeout=60 * 25):
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout,
                       stdin=subprocess.DEVNULL)
    out = (p.stdout or "") + (p.stderr or "")
    print(out[-2500:] if len(out) > 2500 else out)
    if p.returncode != 0:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p.stdout

BASE, OVH = "/kaggle/working", "/kaggle/working/OpenVideoHarness"
os.makedirs(BASE, exist_ok=True); os.chdir(BASE)

if not os.path.isdir(OVH):
    run(f"git clone https://github.com/coreselfmastery/OpenVideoHarness.git {OVH}")
else:
    run(f"git -C {OVH} pull --ff-only || true", timeout=120)

run(f'bash "{OVH}/tools/kaggle/session.sh" --no-vlm', cwd=OVH, timeout=60 * 30)   # idempotent

run("nvidia-smi -L || true")                                     # both T4s, by name
run("pip install -q 'diffusers>=0.32' 'transformers>=4.44' accelerate sentencepiece "
    "protobuf imageio imageio-ffmpeg")                           # the diffusion stack
run("python -c \"import torch, diffusers; print('torch', torch.__version__, 'CUDA', "
    "torch.cuda.is_available(), torch.cuda.device_count(), 'GPUs | diffusers', diffusers.__version__)\"")

## 3 · Connect the AI — Groq model check + auto-selection

In [ ]:
import sys, json
sys.path.insert(0, OVH)
from factory.groq_client import GroqClient

groq = None
try:
    key = GROQ_API_KEY
    for path in ("/kaggle/secrets/GROQ_API_KEY",):              # Kaggle Secrets win
        if os.path.exists(path):
            key = open(path).read().strip() or key
    groq = GroqClient(key)
    models = groq.resolve_models(TEXT_MODEL_PREFERENCE, VISION_MODEL_PREFERENCE)
    print("Groq online:", models)
except Exception as e:
    print("Groq offline (films still get made — the blueprint is the source of truth):", e)
    groq = None

## 4 · The film — *The Hour No One Sees*

One blueprint, deterministic on its own: 26 narration lines timed to exactly 180 s,
**36 cinematic shots** (4:59 alarm → the dark street → the gym → the desk → sunrise),
a 69-bar score in D minor at 92 bpm, foley events on the cut points. The Groq director
polishes shot prompts for the diffusion model — every polish validated, blueprint as
fallback.

In [ ]:
from factory import FILMS

selected = FILMS if ENGINE == "all" else FILMS[:1]
for f in selected:
    print(f"{f['slug']:22s} {f['title']:28s} {f['seconds']:.0f}s · "
          f"{len(f['script'])} lines · {len(f['shots'])} shots")

## 5 · 🏭 Run the factory — one film, end to end

In [ ]:
import logging, time
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s: %(message)s")

from pathlib import Path
from factory.pipeline import Factory, FactoryConfig

cfg = FactoryConfig(engine=ENGINE, target_seconds=TARGET_SECONDS, final_tier=FINAL_TIER,
                    draft_qa=DRAFT_VISION_QA and (groq is not None), shots_timeout=6 * 3600,
                    stage_timeout=50 * 60)
factory = Factory(Path(OVH), Path(OVH), Path(f"{BASE}/films-run"), groq=groq, config=cfg)
if not AI_DIRECTOR:
    factory.groq = None

T0 = time.time()
reports = []
for i, film in enumerate(selected, 1):
    print(f"\n{'='*70}\n🎬 film {i}/{len(selected)}: {film['title']}\n{'='*70}")
    rep = factory.make_film(film)
    reports.append(rep)
    qa = (rep.qa or {}).get("overall")
    print(f"\n[{'OK ' if rep.ok else 'FAIL'}] {film['title']} · {rep.seconds/60:.1f} min · "
          f"vision QA: {qa if qa is not None else '—'} · "
          f"final: {rep.files.get('final') or rep.error}")

print(f"\n🏭 factory done: {sum(r.ok for r in reports)}/{len(reports)} films in "
      f"{(time.time()-T0)/60:.1f} min")

## 6 · Results — the report

In [ ]:
import json
from pathlib import Path

for r in reports:
    print(f"\n{r.title} — {'OK' if r.ok else 'FAIL'} ({r.seconds/60:.1f} min)")
    for k, v in r.stages.items():
        print(f"  {k:12s} {'ok ' if v['ok'] else 'ERR'} {v['s']:8.1f}s" +
              ("  " + v.get("error", "")[:120] if not v["ok"] else ""))
    for n in r.notes:
        print("  ·", n)
    print("  files:", {k: v for k, v in r.files.items() if v})

## 7 · Package the outputs

In [ ]:
import json, shutil, subprocess
from pathlib import Path

OUT = Path("/kaggle/working/discipline-films")
OUT.mkdir(parents=True, exist_ok=True)

for r in reports:
    if not r.ok:
        continue
    final = Path(r.files["final"])
    shutil.copy2(final, OUT / final.name)                       # the master
    meta = OUT / f"{r.slug}-metadata.json"
    meta.write_text(json.dumps(r.metadata, indent=1, ensure_ascii=False))
    if r.files.get("delivered"):
        d = Path(r.files["delivered"])
        if d.exists():
            shutil.copy2(d, OUT / d.name)

p = subprocess.run(["bash", "-c", f"ls -la {OUT} && du -sh {OUT}"], capture_output=True, text=True)
print(p.stdout)
print("\n✅ done — grab the MP4 from the notebook's Output panel "
      "(Save Version → Save & Run All, then the output tab).")

## Notes for the next run

- **Re-runs are cheap-ish**: the diffusion workers skip shots whose clips already exist
  (`projects/<slug>/clips/`), so an interrupted run resumes. Delete the project folder to
  start over. The HF model cache persists in `/root/.cache/huggingface` for the session.
- **The engine ladder**: LTX-Video (both T4s) → per-shot retry at lower res/steps →
  kenburns still+camera-move fill for any straggler → the film always ships.
- No Groq key? Fine — the blueprints ship as-is (the director is pure upside).
- Want the CPU-only look (AI stills + camera moves)? `ENGINE = "kenburns"`.